Markdown
# Evidencia de Aprendizaje 2: Despliegue y Gobierno de una Infraestructura de Datos en la Nube

---

## Sección 1: Diagrama de Arquitectura y Responsabilidades

### 1.1 Diagrama del Flujo de Datos



 ### 1.1 Diagrama del flujo de datos
* 📥 **Fuentes de Datos:** Clickstream JSON, CSVs de reservas y usuarios. 
* └──► **CAPA BRONCE(Ingest Raw):** Almacenamiento en volumenes / Delta Lake sin modificaciones.
* └──► **CAPA PLATA(Limpieza): ** Desanidacion de JSON, tipado de datos y estructuracion relacional.
* └──► **CAPA ORO(Curada):** Tabla `oro_consolidado` con datos agregados y listos para negocio. 
* └──► **Consumo / Analytics: **  Dashboard en Databricks SQL y consultas analiticas.

### 1.2 Responsabilidades de Administracion (PaaS VS Usuario)

* **Administrado por DataBricks (Proveedor PaaS): **
- Mantenimiento del cluster, aprovisionamiento de hardware y virtualizacion.
- Parches de seguridad del sistema operativo, instalacion del runtime de Spark y manejo de dependencias.

- Alta disponibilidad, escabilidad automatica (*Autocaling*) y gestion de metadatos en Unity Catalog.

* **Administrado por nosotros
 (Usuarios / Data Engineers): **

 - Diseño de la estructura de catalogos, esquemas y tablas (Bronce, Plata, Oro).

 - Logica de transformacion del negocio (Pipelines PySpark / SQL).

 - Politicas de seguridad, control de permisos (`GRANT` / `REVOKE`) y orquestacion de tareas en jobs.


## Seccion 2: Organizacion del Entorno (Unity Catalog)

### 2.1 Estructura del Catalogo y esquemas 
En nuestra infraestructura de Databricks se definio la siguiente organizacion de metadatos:

* ** Catalogo:** `workspace`(Catalogo principal asignado al espacio de trabajo).
*  **Esquemas (Base de Datos):**

- `mi_lakehouse` / `bronce`: Almacena las tablas de ingesta en bruto (`bronce_bookings`, `bronce_users`, `bronce_reviews`, `bronce_payments`).
  - `silver`: Destinado al procesamiento intermedio y limpieza de datos.
  - `default` / `gold`: Aloja las tablas curadas finales para el negocio, principalmente `oro_consolidado`.

### 2.2 Justificacion del Criterio de Organizacion:

1. **Aislamiento de Capas: **Se separan los datos crudos (`bronce`) de los datos curados (`gold`) para evitar que consultas analiticas o usuarios finales afecten o modifiquen accidentalmente la informacion fuente.
2. **Control de Acceso Centralizado:** Permite aplicar polticas de seguridad diferenciadas por esquema 
3. ** Escabilidad y Mantenimiento:** Facilita la localizacion de objetos dentro de Unity Catalog y simplifica la gestion del linaje de datos 

## Sección 3: Gobierno de Datos y Seguridad (RBAC & Permisos)

### 3.1 Sentencias DCL Ejecutadas
Para garantizar un acceso controlado a las capas de Lakehouse, se ejecutaron las siguientes sentencias SQL dentro del entorno de Unity Catalog:

` ` `sql
-- 1. Permiso de lectura en la capa Oro para Analistas/Usuarios generales
GRANT SELECT ON SCHEMA gold TO `account users`;

-- 2. Permiso de administración y modificación en la capa Bronce para Ingenieros de Datos
GRANT USE SCHEMA, MODIFY ON SCHEMA bronce TO `account users`;

-- 3. Verificación de permisos aplicados sobre el esquema
SHOW GRANTS ON SCHEMA gold;
` ` `

### 3.2 Matriz de Control de Acceso basado en roles (RBAC)
A continuacion se especifica la matriz de permisos para los roles principales dentro de la organizacion de Wanderbricks:


| Rol | Capa Bronce | Capa Plata | Capa Oro | Justificación de Negocio |
| :--- | :--- | :--- | :--- | :--- |
| **Ingeniero de Datos (Data Engineer)** | `ALL PRIVILEGES` / `MODIFY` | `ALL PRIVILEGES` | `ALL PRIVILEGES` | Administra la ingesta, limpieza y transformación del pipeline completo. |
| **Analista de Datos (Data Analyst)** | `NO ACCESS` | `READ_METADATA` | `SELECT` | Consulta datos consolidados para reportes sin riesgo de alterar la información cruda. |
| **Auditor / Cumplimiento** | `READ_METADATA` | `READ_METADATA` | `READ_METADATA` | Revisa metadatos, linaje y trazabilidad de datos personales sin acceso directo a edición. |

## Seccion 4: Linaje de Datos (Catalog Explorer)

### 4.1 Analisis y Trazabilidad del Pipeline
Desde la interfaz de **Catalog Explorer** en Unity Catalog, se verifico el grafico de linaje (*Lineage Graph*) para la tabla curada `oro_consolidado`:

*  **Tabla Destino (Downstream):** `workspace.default.oro_consolidado`
- `workspace.mi_lakehouse.bronce_bookings`
- `workspace.mi_lakehouse.bronce_users`
- `workspace.mi_lakehouse.bronce_reviews`
- `workspace.mi_lakehouse.bronce_payments`

### 4.2 Importancia del Linaje en el Gobierno de Dato
1. **Impacto de Cambios (Impact Analysis):** Permite evaluar con precision que tablas finales se ven afectadas si se modifica el esquema o la estructura de las crudas en la capa `bronce`.

2. **Auditoria y confianza:** Garantiza la transferencia total sobre el origen de cada columna, asegurando que las transformaciones aplicadas para el consumo analitico sean trazables.

3. **Gestion de Calidad:** Facilita la deteccion de fallos o inconsistencias en los datos rastreando la cadena de procesamiento hacia atras.



## Sección 5: Orquestación y Automatización (Databricks Workflows)

### 5.1 Configuración del Job
Para automatizar el pipeline de datos de Wanderbricks se configuró un flujo de trabajo (*Job*) en Databricks Workflows:

* **Nombre del Job:** `New Job 2026-09-20`
* **Tareas Encadenadas (Task Dependencies):**
  1. `01_Ingesta_Plata`: Ejecuta la lectura, limpieza y estructuración inicial desde los volúmenes a las tablas bronce y plata.
  2. `02_Capa_Oro`: Procesa las transformaciones finales y consolida la información en la tabla `oro_consolidado`.
* **Programación (Schedule):** Ejecución diaria (*Daily*) automatizada.

### 5.2 Monitoreo y Evidencia de Ejecución
* **Registro de Corrida:** Se verificó en la pestaña **Runs** el estado operativo del pipeline.
* **Resultado:** Ejecución exitosa con estado **`Succeeded`** (marcado con indicador verde).
* **Gestión de Errores:** En caso de fallas en alguna tarea intermediaria, Databricks detiene el flujo secuencial y genera alertas de auditoría.

## Sección 6: Comparativa IaaS vs PaaS vs SaaS y Análisis de Infraestructura

### 6.1 Especificación del Equivalente en Infraestructura como Servicio (IaaS)
Si este entorno se desplegara manualmente sobre máquinas virtuales (ej. AWS EC2 o Azure VMs) en lugar de utilizar Databricks (PaaS), se requeriría el siguiente diseño:

* **Arquitectura de Hardware y Nodos:**
  - **Nodo Master / NameNode:** 1 VM (4 vCPU, 16 GB RAM) para coordinar el cluster.
  - **Nodos Worker / DataNodes:** 2 VMs (8 vCPU, 32 GB RAM cada una) para procesamiento distribuido.
* **Sistema Operativo:** Ubuntu Server 22.04 LTS.
* **Stack de Software a Instalar y Configurar:**
  - Apache Spark, Apache Hadoop / HDFS, Hive Metastore, Java OpenJDK 11, Python 3.10, JupyterHub / Apache Zeppelin.
* **Red y Seguridad:**
  - Configuración manual de VPC, subredes públicas/privadas, Security Groups (puertos 8080, 7077, 4040) y llaves SSH.
* **Esfuerzo Operativo Estimado:**
  - **Puesta en marcha:** 2 a 3 semanas (configuración de red, instalación del stack, tuning de Spark y cluster).
  - **Operación continua:** ~15 horas semanales asignadas a parches del SO, monitoreo de disco/memoria, gestión de caídas de nodos y scripts de escalado.

---

### 6.2 Matriz Comparativa de Modelos de Servicio

| Criterio | IaaS (Máquinas Virtuales) | PaaS (Databricks) | SaaS (Power BI / Snowflake) |
| :--- | :--- | :--- | :--- |
| **Control** | Total sobre el SO, puertos y software. | Alto sobre el código y datos; medio sobre infraestructura. | Bajo; limitado a opciones de la aplicación. |
| **Tiempo al 1.er Resultado** | Muy Alto (Semanas de instalación). | Bajo (Minutos / Horas). | Inmediato. |
| **Esfuerzo Operativo** | Muy Alto (Parches, nodos, red manual). | Bajo (Mantenimiento gestionado por el proveedor). | Nulo (Totalmente gestionado). |
| **Costo** | Fijo (VMs encendidas 24/7). | Variable / Eficiente (Pago por uso / Serverless). | Basado en suscripción / licencias. |
| **Escalabilidad** | Compleja (Requiere scripts manuales). | Nativa y Automática (*Autoscaling*). | Automática e Inmediata. |
| **Gobierno de Datos** | Manual (Configurar Apache Ranger/LDAP). | Integrado e Inmediato (Unity Catalog). | Integrado en la aplicación. |

---

### 6.3 Conclusion del Caso Wanderbricks
Para la plataforma analitica de **Wanderbricks**, la eleccion de un modelo **PaaS (Databricks)** es ampliamente superior a un despliegue en **IaaS**:

1. **Eficiencia de Recursos:** Aunque IaaS ofrece control absoluto sobre las maquinas virtuales, el costo operativo y el tiempo requerido para administrar clusters desviaria la atencion del equipo de ingenieria.  

2. **Valor del Negocio:** PasaS permite centrar el 100% del esfuerzo en la contruccion de los pipeline y el analisis de datos desde el primer dia

3. **Gobierno y Seguridad:** La integracion nativa de **Unity Catalog** en PaaS simplifica la gestion de permisos.

## Seccion 7: Declaracion de Trabajo e Integrantes 

### Integrantes de del Grupo: 
* **Andres Felipe Agudelo Salazar**

### Declaracion de Aportes y Roles
En cumplimiento con la regla de integridad de la evidencia de Aprendizaje 2, se detalla el trabajo de manera individual realizado:

- Configuración y estructuración del entorno en **Unity Catalog** (Creación de esquemas `bronce`, `silver`, `gold`).
  - Definición y ejecución de la política de permisos mediante sentencias SQL DCL (`GRANT SELECT`, `GRANT USE SCHEMA, MODIFY`, `SHOW GRANTS`).
  - Configuración y ejecución exitosa del pipeline secuencial mediante **Databricks Workflows** (`Job 2026-09-20` con estado `Succeeded`).
  - Verificación y documentación del gráfico de linaje (*Lineage Graph*) desde **Catalog Explorer**.
  - Redacción del análisis comparativo de arquitectura (IaaS vs PaaS vs SaaS) y sustentación en video.



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronce;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;


In [0]:
%sql
-- 1. PERMISO DE LECTURA DE CAPA GOLD PARA EL GRUPO AANALISTA
GRANT SELECT ON SCHEMA gold TO `account users`;

-- 2. PERMISO DE MODIFICACION EN LA CAPA BRONCE PARA LOS INGENIEROS
GRANT USE SCHEMA, MODIFY ON SCHEMA bronce TO `account users`;

-- 3. PODER MOSTRAR LOS PERMISOS OTORGADOS SOBRE LA CAPA ORO
SHOW GRANTS ON SCHEMA gold;


Principal,ActionType,ObjectType,ObjectKey
account users,SELECT,SCHEMA,workspace.gold


# Evidencia de Aprendizaje 2: Despliegue, Orquestación y Gobierno de Datos Wanderbricks

## 1. Justificación de la Arquitectura Medallón y Unity Catalog

- **Capa Bronce (Raw):** Ingesta cruda de logs y eventos (`clickstream`) preservando la estructura JSON/anidada original para auditoria sin perdida de informacion

- **Capa Plata (Cleansed): ** Desanidacion de JSONS, tipado correcto de datos y estructuracion relacional para consultas operacionales limpias

- **Capa Oro (Curated):** Tablas consolidadas con agregaciones de negocio (`oro_consolidado`) diseñadas para consumo directo por analistas y dashboards.

- **Gobierno con Unity Catalog: ** Permite control de acceso centralizado mediante esquemas explicitos, trazabilidad de linaje (visibilidad de origen y destino ) y aislamiento de entornos .

## 2. Definición del Control de Acceso basado en Roles (RBAC)
Para garantizar la seguridad de la información de Wanderbricks, se estructuran tres roles principales:

1. **Ingeniero de Datos (Data Engineer): **
- *Permisos:* `ALL PRIVILEGES` o `CREATE`, `SELECT`, `MODIFY` sobre esquemas Bronce, Plata y Oro.

- *Justificacion:* Administra los pipelines ETL, creacion de tablas y mantenimiento de la infraestructura.

**Cientifico / Analistas de Datos (Data Analyst):**
- *Permisos:* `SELECT` exclusivo sobre el esquema `gold` y tablas consolidades.
- *Justificacion:* Consume informacion tratada para modelos y reportes sin riesgo de alterar o eliminar datos crudos.

**Auditor de Cumplimiento (Auditor / Compliance): **
- *Permisos:* `READ_METADATA` y consulta de logs de auditoria en Unity Catalog.
- *Justificacion:* Monitorea el linaje y accesos a datos personales (`email_usuario`) sin modificar reglas de negocio.

## 3. Automatización de Flujos con Databricks Workflows (Jobs)

- **Orquestacion:** Creacion de un job secuencial con dependencia explicitas (`01_Ingesta_Plata` -> `02_Capa_Oro`).

- **Frecuencia:** Programacion diaria (Schedule: Daily) para asegurar la actualizacion de los datos analiticos cada 24 horas.

- **Tolerancia a Fallos:** Registros centralizados en la pestaña *Runs* para trazabilidad operacional y monitoreo de estados de ejecucion.